# PhyDG-Slide 1/2: prepare the composite cache (run once, CPU is enough)

Settings (right panel): **Accelerator: None** (saves GPU quota), **Internet: On** (needs a phone-verified Kaggle account).

Then click **Save Version -> Save & Run All (Commit)**. It runs on Kaggle's servers, so you can close the laptop.
It takes about 2-3 hours: it streams about 39 GB from Hugging Face and writes `cache.tar` (about 3.5 GB) to the output.

When it has finished, open notebook 2 (`kaggle_2_train`), click **Add Input -> Your Work -> this notebook**, and its output is attached.

In [ ]:
import os, subprocess
REPO = "/tmp/repo"
BRANCH = "claude/phydg-slide"
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://github.com/abhishekjadhavcs-afk/abhishekjadhavcs-afk-repo {REPO}
%cd {REPO}
!git pull -q
!pip install -q xarray h5netcdf netCDF4
!df -h /tmp /kaggle/working | cat
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "no GPU (fine for the prepare notebook)"

In [ ]:
# Stream the archives; keep only the 4,988 S12LS-LD patches (~15 GB) on local disk
!python scripts/download_s2.py --out /tmp/data/data_harmonized

In [ ]:
!python scripts/build_cache.py --data-root /tmp/data/data_harmonized --out /tmp/cache --workers 4
!rm -rf /tmp/data

In [ ]:
# One uncompressed tar instead of 4,988 small files: faster to save and to attach
!tar -cf /kaggle/working/cache.tar -C /tmp cache && ls -lh /kaggle/working/cache.tar
!python -c "import tarfile; n=sum(m.name.endswith('.npz') for m in tarfile.open('/kaggle/working/cache.tar')); print(n, 'patches'); assert n == 4988"